# 7. *Tokenização* / Vetorização

Este notebook tem como objetivo realizar a vetorização/tokenização de enunciados e alternativas utilizando _embeddings_ de palavras. Ele inclui etapas de limpeza de dados, vetorização e armazenamento dos resultados para uso posterior em análises ou modelos de aprendizado de máquina.


In [1]:
# Importando Dependências para Vetorização
import pandas as pd
import numpy as np
import re
from gensim.models import KeyedVectors

In [3]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
c = np.array([a, b])  # Cria uma matriz 2x3
np.mean(c, axis=0)

array([2.5, 3.5, 4.5])

In [ ]:
enem_df = pd.read_csv("./data/final/cleaned_enem_data.csv")
enem_df.head()

,numero_questao,enunciado,alternativas,gabarito,ano,gabarito_texto,distratores,enunciado_tokens,gabarito_tokens,distratores_tokens,dificuldade
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,C,2009,"reduzir o desmatamento, mantendo-se, assim, o ...",reduzir o calor irradiado pela Terra mediante ...,atmosfera terrestre composta gases nitrogênio ...,reduzir desmatamento mantendo assim potencial ...,reduzir calor irradiado terra mediante substit...,-1.70677
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,D,2009,Estimativa de tempo necessário para metaboliza...,Concentração média de álcool no sangue ao long...,analise figura supondo necessário dar título f...,estimativa tempo necessário metabolizar difere...,concentração média álcool sangue longo dia var...,0.62043
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",A,2009,"induzir a imunidade, para proteger o organismo...",ser capaz de alterar o genoma do organismo por...,estima atualmente mundo milhões pessoas infect...,induzir imunidade proteger organismo contamina...,capaz alterar genoma organismo portador induzi...,2.07704
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,B,2009,os genótipos idênticos e os fenótipos diferentes.,os genótipos e os fenótipos idênticos.; difere...,experimento preparou conjunto plantas técnica ...,genótipos idênticos fenótipos diferentes,genótipos fenótipos idênticos diferenças genót...,0.11500
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,E,2009,"Kepler apresentou uma teoria científica que, g...","Ptolomeu apresentou as ideias mais valiosas, p...",linha tradição antiga astrônomo grego ptolomeu...,kepler apresentou teoria científica graças mét...,ptolomeu apresentou ideias valiosas serem anti...,0.21694


---

## 7.1. Word2Vec


In [6]:
# Modelo de Embedding: Word2Vec NILC
# Download disponível em:
# http://nilc.icmc.usp.br/nilc/index.php/repositorio-de-word-embeddings-do-nilc
model = KeyedVectors.load_word2vec_format("/content/drive/MyDrive/cbow_s300.txt")

In [48]:
def generate_word2vec_embeddings(data, model):
    not_found_words = set()
    result_embeddings = []

    # Precompilar regex para remover pontuações
    clean_ponctuation = re.compile(r"[.,:;()]")

    for item in data:
        word_vectors = []

        # Remove pontuações e divide em palavras
        words = clean_ponctuation.sub("", str(item)).split()

        for word in words:
            word_lower = word.lower()

            if word_lower in model:
                word_vectors.append(model[word_lower])
            else:
                not_found_words.add(word_lower)
        if word_vectors:
            # Calcula a média dos vetores
            mean_vector = np.mean(word_vectors, axis=0)
            result_embeddings.append(mean_vector)
        else:
            # Se não encontrou nenhuma palavra, adiciona um vetor nulo
            result_embeddings.append(np.zeros(model.vector_size))

    return result_embeddings, not_found_words

### 7.1.1. Vetorização dos Enunciados


In [ ]:
enem_df["enunciado_embbedings_word2vec"], not_found_words_enunciado = (
    generate_word2vec_embeddings(enem_df["enunciado_tokens"], model)
)

In [50]:
len(not_found_words_enunciado)

155

### 7.1.2. Vetorização dos Gabaritos


In [ ]:
enem_df["gabarito_embbedings_word2vec"], not_found_words_gabarito = (
    generate_word2vec_embeddings(enem_df["gabarito_tokens"], model)
)

In [52]:
not_found_words_gabarito

{'anfifílica',
 'catalisem',
 'eflluente',
 'fotoimunoterapia',
 'fácia',
 'hexan',
 'leisnmaniose',
 'monofluoracético',
 'nitratação',
 'ondana',
 'penicilamina',
 'tetrassômicos',
 'trissômicos'}

### 7.1.3. Vetorização dos Distratores


In [ ]:
enem_df["distratores_embbedings_word2vec"], not_found_words_distratores = (
    generate_word2vec_embeddings(enem_df["distratores_tokens"], model)
)

In [54]:
not_found_words_distratores

{'acondicionantes',
 'amonificação',
 'anfifílicos',
 'anfotérica',
 'apocinaceae',
 'borrifaria',
 'canade',
 'caramelizam',
 'cicloexanol',
 'codominante',
 'convergidos',
 'cromatofilia',
 'ddois',
 'eterificação',
 'fotoquimicamente',
 'fácia',
 'gãs',
 'hemozoínas',
 'hexan',
 'hexanal',
 'hexanoico',
 'interpopulacional',
 'linearizadas',
 'microvespa',
 'monofluoracético',
 'nitrosação',
 'owudomr',
 'penicilamina',
 'pirossulfato',
 'polialélico',
 'poligênico',
 'polipirrol',
 'polipoidia',
 'poliuretana',
 'precessionar',
 'resfriaria',
 'salinificação',
 'sinfilia',
 'solubilizado',
 'tetrassômicos',
 'trissômicos',
 'volatilizando',
 'vígula'}

### 7.1.4. Salvando Embeddings


In [ ]:
enem_df.to_csv("../data/final/enem_data_embeddings.csv")

In [ ]:
enem_df.to_pickle("../data/final/enem_data_embeddings.pkl")

In [6]:
df = pd.read_pickle("../data/final/enem_data_embeddings.pkl")
df.head()

,numero_questao,enunciado,alternativas,gabarito,ano,gabarito_texto,distratores,enunciado_tokens,gabarito_tokens,distratores_tokens,dificuldade,enunciado_embbedings_word2vec,gabarito_embbedings_word2vec,distratores_embbedings_word2vec
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,C,2009,"reduzir o desmatamento, mantendo-se, assim, o ...",reduzir o calor irradiado pela Terra mediante ...,atmosfera terrestre composta gases nitrogênio ...,reduzir desmatamento mantendo assim potencial ...,reduzir calor irradiado terra mediante substit...,-1.70677,"[-0.0016308315, -0.057879616, -0.085349284, 0....","[0.007537251, -0.049133625, 0.11472875, -0.115...","[0.03476311, -0.035221867, -0.00517779, -0.128..."
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,D,2009,Estimativa de tempo necessário para metaboliza...,Concentração média de álcool no sangue ao long...,analise figura supondo necessário dar título f...,estimativa tempo necessário metabolizar difere...,concentração média álcool sangue longo dia var...,0.62043,"[-0.026465332, -0.030027837, 0.028988913, 0.14...","[0.07856357, -0.022353431, -0.12115115, 0.0183...","[0.0763594, -0.11076818, 0.0048947046, 0.04739..."
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",A,2009,"induzir a imunidade, para proteger o organismo...",ser capaz de alterar o genoma do organismo por...,estima atualmente mundo milhões pessoas infect...,induzir imunidade proteger organismo contamina...,capaz alterar genoma organismo portador induzi...,2.07704,"[0.027941352, 0.008942129, 0.08566157, 0.02149...","[0.097563, -0.0034619968, 0.08206, -0.109324, ...","[0.03503356, -0.010700853, 0.008868624, -0.027..."
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,B,2009,os genótipos idênticos e os fenótipos diferentes.,os genótipos e os fenótipos idênticos.; difere...,experimento preparou conjunto plantas técnica ...,genótipos idênticos fenótipos diferentes,genótipos fenótipos idênticos diferenças genót...,0.11500,"[-0.027700324, -0.014135911, -0.03359886, -0.0...","[0.098122, 0.0499915, -0.214217, -0.0571225, -...","[0.0143338675, -0.014280667, -0.21914314, 0.08..."
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,E,2009,"Kepler apresentou uma teoria científica que, g...","Ptolomeu apresentou as ideias mais valiosas, p...",linha tradição antiga astrônomo grego ptolomeu...,kepler apresentou teoria científica graças mét...,ptolomeu apresentou ideias valiosas serem anti...,0.21694,"[-0.027011229, 0.023866067, -0.08451317, 0.043...","[-0.031293802, 0.018204402, -0.0655475, 0.0612...","[-0.0040200884, 0.03563236, -0.038875517, 0.06..."
